##### GOLD датасет состоит из:
- примерно 800 предложений, которые разметил Разметчик
    - вычислить их легко - там где будет `spans_rus` или `relations_rus` непустой
- предложений, которые были отправлены на простую проверку перевода
    - предварительно собрать ID предложений, отправленных на проверку перевода
    - собрать данные по этим ID

##### Остальное - отправим на перевод через LLM


Предполагается, что оригинальный датасет лежит в директории `../hf_deft_ru/`

In [2]:
import json
from tqdm import tqdm
import logging
from datasets import Dataset
import datetime


Настройка логгера, для вывода подробной информации работа скрипта в файл `log.log`

При этом основной вывод прогрессбара от `tqdm` будет осуществляться в `Out` ячейку

In [3]:
# Настройка логгера для записи в файл
logger = logging.getLogger()
logger.setLevel(logging.DEBUG)

# Создание обработчика для записи в файл
file_handler = logging.FileHandler('log_make_data_for_translation.log')
file_handler.setLevel(logging.DEBUG)

# Создание форматтера для сообщений
FORMAT = "%(asctime)s %(levelname)s [%(process)s]: %(name)s: %(message)s"
formatter = logging.Formatter(FORMAT)
file_handler.setFormatter(formatter)

# Добавление обработчика к логгеру
logger.addHandler(file_handler)

In [4]:
# orig files
dev_file = "../hf_deft_ru/dev.json"
train_file = "../hf_deft_ru/train.json"

In [5]:
gold_file = "gold/rudeft_gold.json"

In [13]:
def make_dataset_for_translation(hf_file_dev, hf_file_train, gold_file, new_file_dev, new_file_train):
    """ Функция создания датасета для перевода в HF (Hugging Face) формате.

    Создаем датасет из тех предложений, которых нет в GOLD датасете.

    
    :param hf_file_dev: путь к dev файлу датасета в HF формате.
    :param hf_file_train: путь к train файлу датасета в HF формате.
    :param gold_file: путь к файлу GOLD датасета.
    :param new_file_dev: путь к новому dev файлу датасета в HF формате, из которого будут удалены GOLD примеры.
    :param new_file_train: путь к новому train файлу датасета в HF формате, из которого будут удалены GOLD примеры.

    """
    # считываем данные из hf датасета   
    hf_data_dev = Dataset.from_json(hf_file_dev)        # list of dicts (dict = one line from hf dataset)
    hf_data_dev = hf_data_dev.to_list()
    hf_data_train = Dataset.from_json(hf_file_train)        # list of dicts (dict = one line from hf dataset)
    hf_data_train = hf_data_train.to_list()
    hf_gold = Dataset.from_json(gold_file)        # list of dicts (dict = one line from hf dataset)
    hf_gold = hf_gold.to_list()

   
    # создаем множество ID из голд датасета
    gold_ids = set()
    for index, elem in tqdm(enumerate(hf_gold), desc="Collect gold ids"):
        gold_ids.add(elem["id"])


    logging.info(f"Processing DEV...")
    new_dev = []
    for hf_elem in tqdm(hf_data_dev, desc="Processing HF DEV data"):

        hf_id = hf_elem['id']

        if hf_id in gold_ids:
            logging.info(f"HF. DEV_ID: '{hf_id}' is in gold_ids! Skip!")
            continue
        new_dev.append(hf_elem)

    logging.info(f"Processing TRAIN...")
    new_train = []
    for hf_elem in tqdm(hf_data_train, desc="Processing HF TRAIN data"):

        hf_id = hf_elem['id']

        if hf_id in gold_ids:
            logging.info(f"HF. TRAIN_ID: '{hf_id}' is in gold_ids! Skip!")
            continue
        new_train.append(hf_elem)

    
    # Записываем данные в HF датасет
    new_dev_dataset = Dataset.from_list(new_dev)
    new_dev_dataset.to_json(new_file_dev)

    new_train_dataset = Dataset.from_list(new_train)
    new_train_dataset.to_json(new_file_train)

    logging.info(f"Length new DEV: {len(new_dev_dataset)}")
    logging.info(f"Info new DEV: {new_dev_dataset}")

    logging.info(f"Length new TRAIN: {len(new_train_dataset)}")
    logging.info(f"Info new TRAIN: {new_train_dataset}")

In [14]:
new_dev_file = "../hf_rudeft_for_translation/dev.json"
new_train_file = "../hf_rudeft_for_translation/train.json"


In [15]:
make_dataset_for_translation(dev_file, train_file, gold_file, new_dev_file, new_train_file)

Collect gold ids: 1070it [00:00, 918711.42it/s]
Creating json from Arrow format: 100%|██████████| 24/24 [00:00<00:00, 36.49ba/s]


In [10]:
!grep "Processed:" log_make_data_for_translation.log 